In [1]:
import tubesml as tml
import pandas as pd
import numpy as np

from source.report import _point_to_proba

from sklearn.metrics import brier_score_loss, mean_squared_error

from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge, LogisticRegression, Lasso
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import optuna
from optuna.samplers import TPESampler

In [2]:
df = pd.read_csv('data/processed/women_training.csv')

N_FOLDS = 5
kfolds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=13)

df_train, df_test = tml.make_test(df, test_size=0.2, random_state=34)

DROP = ["target", "target_points", "ID", "DayNum", "Team1", "Team2",
        'T1_Loc', 'T2_Loc',
                "T1_region", "T2_region", "Season", "delta_Loc",
                "Season", "competitive", "competitive_score",
                "delta_def_rating_diff", "delta_impact_diff",
                "T1_def_rating_diff", "T2_def_rating_diff"]

df_train.head()

df_train = df.copy()

## Feats cats

In [3]:
all_feats = [c for c in df_train if c not in DROP]
all_feats

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [4]:
all_delta = [c for c in df_train if c not in DROP and "delta" in c]

all_delta

['delta_Ast',
 'delta_Ast_TO_ratio',
 'delta_Ast_TO_ratio_diff',
 'delta_Ast_diff',
 'delta_Away',
 'delta_Blk',
 'delta_Blk_diff',
 'delta_DR',
 'delta_DR_diff',
 'delta_DR_opportunity',
 'delta_DR_opportunity_diff',
 'delta_Eff_FG_perc_diff',
 'delta_FG3_ratio',
 'delta_FG3_ratio_diff',
 'delta_FGA',
 'delta_FGA2',
 'delta_FGA2_diff',
 'delta_FGA3',
 'delta_FGA3_diff',
 'delta_FGA_diff',
 'delta_FGM',
 'delta_FGM2',
 'delta_FGM2_diff',
 'delta_FGM3',
 'delta_FGM3_diff',
 'delta_FGM_diff',
 'delta_FGM_no_ast',
 'delta_FGM_no_ast_diff',
 'delta_FTA',
 'delta_FTA_diff',
 'delta_FTM',
 'delta_FTM_diff',
 'delta_N_wins',
 'delta_OR',
 'delta_OR_diff',
 'delta_OR_opportunity',
 'delta_OR_opportunity_diff',
 'delta_OT_win',
 'delta_PF',
 'delta_PF_diff',
 'delta_Score',
 'delta_Score_diff',
 'delta_Stl',
 'delta_Stl_diff',
 'delta_TO',
 'delta_TO_diff',
 'delta_TO_perposs',
 'delta_TO_perposs_diff',
 'delta_Tot_Reb',
 'delta_Tot_Reb_diff',
 'delta_True_shooting_perc_diff',
 'delta_def_ratin

In [5]:
no_delta = [c for c in df_train if c not in DROP and "delta" not in c]
no_delta

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [6]:
seeds = [c for c in df_train if c not in DROP and "Seed" in c] + [c for c in df_train if "quality" in c] + [c for c in df_train if "stage" in c] + [c for c in df_train if "elo" in c]
seeds

['T1_Seed',
 'T2_Seed',
 'delta_Seed',
 'T1_quality',
 'T2_quality',
 'delta_quality',
 'stage_Round1',
 'stage_Round2',
 'stage_Round3',
 'stage_Round4',
 'stage_final',
 'stage_finalfour',
 'stage_impossible',
 'T1_elo',
 'T2_elo',
 'delta_elo']

In [7]:
no_seeds = [c for c in df_train if c not in DROP and "Seed" not in c]
no_seeds

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [8]:
feats_dict = {"all_feats": all_feats,
              "all_delta": all_delta, "no_delta": no_delta, "no_seeds": no_seeds, "seeds": seeds}

# Points predictions

## LGBM

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMRegressor(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                              learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="l2")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "l2"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

[I 2026-03-07 16:56:54,590] A new study created in memory with name: no-name-91533aa7-0b86-47df-bde0-6c85bff439e2
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
in

Number of finished trials: 2000
Best trial: {'max_depth': 144, 'num_leaves': 53, 'reg_lambda': 8.885726284153476, 'reg_alpha': 2.600387041450046, 'colsample_bytree': 0.9074559985648463, 'subsample': 0.8299784877820622, 'min_child_weight': 192.85514300845676, 'feats': 'all_delta', 'clip_val': 21, 'padd': 0.041387752386437195}


In [11]:
0.183152

0.183152

In [12]:
study.trials_dataframe().sort_values('value', ascending=True).head(20)

,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
1928,1928,0.139372,2026-03-07 17:32:44.332359,2026-03-07 17:33:02.081328,0 days 00:00:17.748969,21,0.907456,all_delta,144,192.855143,53,0.041388,2.600387,8.885726,0.829978,COMPLETE
1022,1022,0.139501,2026-03-07 17:16:21.949221,2026-03-07 17:16:36.034185,0 days 00:00:14.084964,23,0.919262,all_delta,127,188.754273,45,0.047955,5.817860,0.034770,0.416891,COMPLETE
1033,1033,0.139505,2026-03-07 17:16:31.696308,2026-03-07 17:16:47.126982,0 days 00:00:15.430674,24,0.900083,all_delta,135,189.156186,41,0.044805,5.683902,5.519723,0.411037,COMPLETE
1131,1131,0.139554,2026-03-07 17:18:12.895593,2026-03-07 17:18:29.801655,0 days 00:00:16.906062,22,0.854340,all_delta,132,191.938940,56,0.048039,6.246979,1.280587,0.433228,COMPLETE
1145,1145,0.139592,2026-03-07 17:18:28.483192,2026-03-07 17:18:42.011317,0 days 00:00:13.528125,22,0.852450,all_delta,133,192.323410,49,0.043627,2.297060,0.121085,0.432054,COMPLETE
1357,1357,0.139592,2026-03-07 17:22:24.011589,2026-03-07 17:22:39.720609,0 days 00:00:15.709020,21,0.872523,all_delta,109,192.325073,44,0.044191,8.853498,5.577900,0.776342,COMPLETE
1676,1676,0.139599,2026-03-07 17:28:06.224836,2026-03-07 17:28:23.528759,0 days 00:00:17.303923,21,0.872729,all_delta,117,182.164271,51,0.042196,10.478233,9.642000,0.446000,COMPLETE
931,931,0.139606,2026-03-07 17:14:42.957871,2026-03-07 17:14:57.571851,0 days 00:00:14.613980,21,0.900434,all_delta,140,192.108249,50,0.042876,13.523450,4.848467,0.413298,COMPLETE
1576,1576,0.139625,2026-03-07 17:26:21.094384,2026-03-07 17:26:38.442985,0 days 00:00:17.348601,23,0.888195,all_delta,141,194.612995,47,0.045498,2.303123,0.027972,0.421267,COMPLETE
1130,1130,0.139664,2026-03-07 17:18:12.455996,2026-03-07 17:18:26.412927,0 days 00:00:13.956931,22,0.856115,all_delta,133,191.559968,56,0.048364,6.708223,1.937490,0.400572,COMPLETE


## XGBoost

In [13]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBRegressor(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric=mean_squared_error)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {'verbose': False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [14]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2000
Best trial: {'max_depth': 3, 'reg_lambda': 44.062971221405036, 'reg_alpha': 11.413674009182976, 'colsample_bytree': 0.8544560837593326, 'colsample_bylevel': 0.5969234712473717, 'subsample': 0.7813041231083666, 'min_child_weight': 254.60687569207073, 'feats': 'all_delta', 'clip_val': 20, 'padd': 0.030643412875891236}


,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
1870,1870,0.138646,2026-03-07 21:26:54.215934,2026-03-07 21:28:08.148008,0 days 00:01:13.932074,20,0.596923,0.854456,all_delta,3,254.606876,0.030643,11.413674,44.062971,0.781304,COMPLETE
1874,1874,0.138688,2026-03-07 21:27:13.223324,2026-03-07 21:28:18.752762,0 days 00:01:05.529438,20,0.570239,0.840950,all_delta,3,255.061588,0.029089,9.373684,45.719880,0.782092,COMPLETE
1884,1884,0.138750,2026-03-07 21:28:08.154635,2026-03-07 21:29:10.440525,0 days 00:01:02.285890,20,0.579168,0.842101,all_delta,3,244.526456,0.032497,8.353480,43.341902,0.781647,COMPLETE
1924,1924,0.139044,2026-03-07 21:32:21.632753,2026-03-07 21:33:15.013291,0 days 00:00:53.380538,20,0.604211,0.845593,all_delta,6,238.626603,0.031591,11.451028,41.412549,0.797028,COMPLETE
1454,1454,0.139156,2026-03-07 20:32:56.090112,2026-03-07 20:33:47.650049,0 days 00:00:51.559937,21,0.611043,0.844253,all_delta,3,259.045718,0.011973,83.629621,11.222347,0.816105,COMPLETE
1648,1648,0.139242,2026-03-07 20:50:11.541375,2026-03-07 20:51:11.498557,0 days 00:00:59.957182,21,0.792809,0.824365,all_delta,3,293.909373,0.033196,4.929587,32.566654,0.889037,COMPLETE
1749,1749,0.139263,2026-03-07 21:03:06.635483,2026-03-07 21:03:51.088764,0 days 00:00:44.453281,21,0.573534,0.834458,all_delta,3,240.364782,0.032967,5.194224,39.775215,0.769315,COMPLETE
1694,1694,0.139337,2026-03-07 20:54:15.168664,2026-03-07 20:55:12.183892,0 days 00:00:57.015228,20,0.590641,0.836086,all_delta,3,297.325631,0.013543,2.067928,34.723636,0.769091,COMPLETE
421,421,0.139364,2026-03-07 18:35:54.857226,2026-03-07 18:36:55.370421,0 days 00:01:00.513195,20,0.506192,0.826772,all_delta,3,277.212424,0.014659,4.250272,58.647561,0.809873,COMPLETE
206,206,0.139390,2026-03-07 18:14:28.886849,2026-03-07 18:15:20.651479,0 days 00:00:51.764630,20,0.656715,0.767686,all_delta,3,189.527791,0.014369,33.110529,13.399360,0.818434,COMPLETE


## Ridge

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Ridge(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

[I 2026-03-07 23:00:34,483] A new study created in memory with name: no-name-31e46ab1-c04a-4899-a0b3-4ae655592049
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
in

Number of finished trials: 2000
Best trial: {'alpha': 21.832977504843903, 'feats': 'all_delta', 'clip_val': 29, 'padd': 0.0057516248250359565}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1318,1318,0.137389,2026-03-07 23:17:00.844203,2026-03-07 23:17:10.363204,0 days 00:00:09.519001,21.832978,29,all_delta,0.005752,COMPLETE
1443,1443,0.137389,2026-03-07 23:18:20.463860,2026-03-07 23:18:31.479294,0 days 00:00:11.015434,21.844903,29,all_delta,0.003638,COMPLETE
1750,1750,0.137389,2026-03-07 23:22:02.359264,2026-03-07 23:22:11.817792,0 days 00:00:09.458528,21.905997,29,all_delta,0.006487,COMPLETE
1502,1502,0.137389,2026-03-07 23:18:55.839459,2026-03-07 23:19:05.495744,0 days 00:00:09.656285,21.920002,29,all_delta,0.003799,COMPLETE
1511,1511,0.137389,2026-03-07 23:18:58.424234,2026-03-07 23:19:08.444121,0 days 00:00:10.019887,21.920907,29,all_delta,0.004553,COMPLETE
1747,1747,0.137389,2026-03-07 23:22:01.743040,2026-03-07 23:22:11.592279,0 days 00:00:09.849239,21.924115,29,all_delta,0.006513,COMPLETE
1929,1929,0.137389,2026-03-07 23:23:59.990756,2026-03-07 23:24:09.965863,0 days 00:00:09.975107,21.933659,29,all_delta,0.007756,COMPLETE
1930,1930,0.137389,2026-03-07 23:24:00.088002,2026-03-07 23:24:10.927615,0 days 00:00:10.839613,21.954954,29,all_delta,0.007475,COMPLETE
1925,1925,0.137389,2026-03-07 23:23:58.137980,2026-03-07 23:24:09.577443,0 days 00:00:11.439463,22.039774,29,all_delta,0.001577,COMPLETE
1873,1873,0.137389,2026-03-07 23:23:13.784114,2026-03-07 23:23:23.407891,0 days 00:00:09.623777,22.085128,29,all_delta,0.003890,COMPLETE


## Lasso

In [11]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Lasso(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [12]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `

Number of finished trials: 2000
Best trial: {'alpha': 0.10384021896343953, 'feats': 'all_delta', 'clip_val': 28, 'padd': 0.01584309458358785}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
900,900,0.137041,2026-03-07 23:46:45.517796,2026-03-07 23:46:57.176854,0 days 00:00:11.659058,0.103840,28,all_delta,0.015843,COMPLETE
1911,1911,0.137043,2026-03-07 23:58:39.957098,2026-03-07 23:58:50.546720,0 days 00:00:10.589622,0.102477,28,all_delta,0.015194,COMPLETE
1861,1861,0.137043,2026-03-07 23:57:50.752824,2026-03-07 23:58:01.550805,0 days 00:00:10.797981,0.107156,29,all_delta,0.013136,COMPLETE
1927,1927,0.137044,2026-03-07 23:58:47.557151,2026-03-07 23:58:58.409184,0 days 00:00:10.852033,0.113699,29,all_delta,0.015580,COMPLETE
1586,1586,0.137045,2026-03-07 23:54:41.549195,2026-03-07 23:54:52.273541,0 days 00:00:10.724346,0.112803,28,all_delta,0.018783,COMPLETE
1991,1991,0.137046,2026-03-07 23:59:34.101832,2026-03-07 23:59:43.787858,0 days 00:00:09.686026,0.110167,27,all_delta,0.015407,COMPLETE
1591,1591,0.137046,2026-03-07 23:54:44.098924,2026-03-07 23:54:55.730848,0 days 00:00:11.631924,0.104979,28,all_delta,0.017893,COMPLETE
902,902,0.137048,2026-03-07 23:46:46.209750,2026-03-07 23:46:57.546556,0 days 00:00:11.336806,0.122213,28,all_delta,0.015872,COMPLETE
1804,1804,0.137049,2026-03-07 23:57:17.918096,2026-03-07 23:57:29.926386,0 days 00:00:12.008290,0.108994,29,all_delta,0.016936,COMPLETE
901,901,0.137050,2026-03-07 23:46:45.594595,2026-03-07 23:46:54.965071,0 days 00:00:09.370476,0.101185,28,all_delta,0.017188,COMPLETE


# Probability Predictions


## LGBM

In [ ]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMClassifier(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                               learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "auc"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 92, 'num_leaves': 51, 'reg_lambda': 44.00536243237047, 'reg_alpha': 3.517393066057428, 'colsample_bytree': 0.4112546404929133, 'subsample': 0.898529310176569, 'min_child_weight': 47.85558942186549, 'feats': 'all_feats'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,state
897,897,0.186719,2026-03-07 04:36:45.036229,2026-03-07 04:37:23.174325,0 days 00:00:38.138096,0.411255,all_feats,92,47.855589,51,3.517393,44.005362,0.898529,COMPLETE
884,884,0.187136,2026-03-07 04:36:16.578492,2026-03-07 04:36:47.198538,0 days 00:00:30.620046,0.407672,all_feats,94,51.406472,46,1.748187,0.867684,0.897356,COMPLETE
887,887,0.187146,2026-03-07 04:36:24.714915,2026-03-07 04:36:58.384573,0 days 00:00:33.669658,0.345501,all_feats,93,53.048506,47,0.033628,47.194002,0.973086,COMPLETE
653,653,0.187184,2026-03-07 04:28:38.906368,2026-03-07 04:29:10.299023,0 days 00:00:31.392655,0.314533,all_feats,88,47.624331,51,1.685361,46.770161,0.932810,COMPLETE
734,734,0.187189,2026-03-07 04:31:29.729335,2026-03-07 04:32:02.983681,0 days 00:00:33.254346,0.347716,all_feats,88,44.749334,51,1.523920,52.038074,0.897639,COMPLETE
516,516,0.187248,2026-03-07 04:24:23.691570,2026-03-07 04:24:53.385911,0 days 00:00:29.694341,0.350284,all_feats,98,44.902064,78,3.232698,49.731937,0.889971,COMPLETE
479,479,0.187258,2026-03-07 04:23:21.947479,2026-03-07 04:23:47.433756,0 days 00:00:25.486277,0.371985,all_feats,160,33.903735,88,0.575452,46.353491,0.964723,COMPLETE
635,635,0.187271,2026-03-07 04:28:01.396043,2026-03-07 04:28:34.786178,0 days 00:00:33.390135,0.317591,all_feats,97,30.360887,45,0.020070,46.551021,0.910511,COMPLETE
645,645,0.187303,2026-03-07 04:28:22.074692,2026-03-07 04:28:54.654826,0 days 00:00:32.580134,0.386529,all_feats,66,46.834857,49,1.479965,47.422417,0.865707,COMPLETE
668,668,0.187320,2026-03-07 04:29:09.552888,2026-03-07 04:29:46.447713,0 days 00:00:36.894825,0.366288,all_feats,50,48.592149,50,1.726721,55.643695,0.859652,COMPLETE


## XGBoost

In [ ]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBClassifier(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {"verbose": False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 108, 'reg_lambda': 30.935315773225398, 'reg_alpha': 8.524310782577258, 'colsample_bytree': 0.4914753840095336, 'colsample_bylevel': 0.3002201650698665, 'subsample': 0.400991101026403, 'min_child_weight': 131.93444263029593, 'feats': 'seeds'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state
914,914,0.188007,2026-03-07 05:22:45.850325,2026-03-07 05:23:13.491791,0 days 00:00:27.641466,0.300220,0.491475,seeds,108,131.934443,8.524311,30.935316,0.400991,COMPLETE
832,832,0.188067,2026-03-07 05:19:56.257032,2026-03-07 05:20:32.375884,0 days 00:00:36.118852,0.353506,0.504133,seeds,97,118.838785,1.598024,37.485915,0.420642,COMPLETE
704,704,0.188109,2026-03-07 05:15:21.517040,2026-03-07 05:15:44.865020,0 days 00:00:23.347980,0.354766,0.548414,seeds,68,16.805789,5.398335,37.906043,0.445922,COMPLETE
799,799,0.188268,2026-03-07 05:17:34.439975,2026-03-07 05:18:11.975474,0 days 00:00:37.535499,0.374607,0.519837,seeds,91,20.994222,3.277896,52.588546,0.409538,COMPLETE
361,361,0.188287,2026-03-07 04:59:18.389655,2026-03-07 05:00:17.856013,0 days 00:00:59.466358,0.308716,0.500086,seeds,90,217.713533,11.935586,36.741048,0.435749,COMPLETE
550,550,0.188340,2026-03-07 05:08:33.017084,2026-03-07 05:09:01.460416,0 days 00:00:28.443332,0.375717,0.485768,seeds,72,211.302444,7.302716,33.351364,0.440366,COMPLETE
675,675,0.188363,2026-03-07 05:14:27.118707,2026-03-07 05:14:57.411538,0 days 00:00:30.292831,0.379720,0.460420,seeds,98,16.621569,1.747527,38.168762,0.403896,COMPLETE
879,879,0.188365,2026-03-07 05:21:58.760370,2026-03-07 05:22:20.385901,0 days 00:00:21.625531,0.361203,0.547310,seeds,114,17.684617,5.505772,37.355211,0.444824,COMPLETE
204,204,0.188371,2026-03-07 04:52:59.533648,2026-03-07 04:53:20.247208,0 days 00:00:20.713560,0.320349,0.498580,seeds,47,154.963562,8.542859,8.548269,0.459351,COMPLETE
618,618,0.188373,2026-03-07 05:10:45.578551,2026-03-07 05:11:09.542916,0 days 00:00:23.964365,0.387443,0.507357,seeds,88,13.490536,8.873495,31.159116,0.444990,COMPLETE


## LogisticRegression

In [9]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        'C': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = LogisticRegression(C=param["C"], random_state=34, max_iter=10000, n_jobs=-1)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1, show_progress_bar=True)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

[I 2026-03-08 18:00:35,571] A new study created in memory with name: no-name-b10f9a6c-2c4b-4365-b37c-3ed3524094e8


  0%|          | 0/2000 [00:00<?, ?it/s]

invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
